In [4]:
import pandas as pd
import os

In [87]:
# Profiles downloaded from https://dataminer2.pjm.com/feed/hrl_load_metered
load_profile_list = []

dir = '../data/iso_load_profiles/raw/pjm'
for filename in os.listdir(dir):
    f = os.path.join(dir, filename)
    df = pd.read_csv(f)
    df = df.loc[df.zone != 'RTO']
    df = (
        df.assign(timestamp=(
            pd.to_datetime(df['datetime_beginning_utc']) + pd.Timedelta(hours=1)
        ))
        .groupby(['timestamp', 'zone'])
        .sum(numeric_only=True)
        .reset_index()
        .drop(columns='is_verified')
        .rename(columns={'zone': 'subba', 'mw': 'value'})
    )
    load_profile_list.append(df)

In [88]:
pjm_load = pd.concat(load_profile_list, ignore_index=True)
pjm_load.to_csv(f"../data/iso_load_profiles/pjm.csv", index=False)

In [158]:
forecast_area_subba_map = {
    'AEP': 'AEP',
    'AP': 'AP',
    'APS': 'AP',
    'COMED': 'CE',
    'DEOK': 'DEOK',
    'DOM': 'DOM',
    'DOMINION': 'DOM',
    'DAY': 'DAY',
    'DAYTON': 'DAY',
    'DUQ': 'DUQ',
    'DUQUESNE': 'DUQ',
    'EKPC': 'EKPC',
    'ATSI': 'ATSI'
}

In [178]:
# Profiles downloaded from https://dataminer2.pjm.com/feed/load_frcstd_hist
forecast_profile_list = []

dir = '../data/iso_load_profiles/raw/pjm_forecast'
for filename in os.listdir(dir):
    f = os.path.join(dir, filename)
    df = pd.read_csv(f)
    df['subba'] = df['forecast_area'].map(forecast_area_subba_map)
    unmapped_areas = df.loc[df.subba.isna()]['forecast_area'].unique().tolist()
    if len(unmapped_areas) > 0:
        print(f"Unmapped areas: {unmapped_areas}")
    df = df.dropna(subset='subba')
    df['evaluation_time'] = pd.to_datetime(
        df['evaluated_at_ept'],
        format="%m/%d/%Y %H:%M:%S %p"
    )
    df['delivery_time'] = pd.to_datetime(
        df['forecast_hour_beginning_ept'],
        format="%m/%d/%Y %H:%M:%S %p"
    )
    df = df.loc[(
        (df.evaluation_time.dt.hour < 11)
        & (df.delivery_time.dt.date - df.evaluation_time.dt.date == pd.Timedelta(days=1))
    )]
    df['timestamp'] = pd.to_datetime(df['forecast_hour_beginning_utc']) + pd.Timedelta(hours=1)
    df = (
        df.sort_values('evaluation_time', ascending=False)
        .drop_duplicates(subset=['subba', 'timestamp'], keep='first')
        .rename(columns={'forecast_load_mw': 'value'})
        [['timestamp', 'subba', 'value']]
    )

    forecast_profile_list.append(df)

Unmapped areas: ['MIDATL', 'RTO', 'AE/MIDATL', 'BG&E/MIDAT', 'DP&L/MIDAT', 'JCP&L/MIDA', 'METED/MIDA', 'MID ATLANT', 'PECO/MIDAT', 'PENELEC/MI', 'PEPCO/MIDA', 'PPL/MIDATL', 'PSE&G/MIDA', 'RECO/MIDAT', 'RTO COMBIN', 'SOUTHERN', 'UGI/MIDATL', 'WESTERN']
Unmapped areas: ['MIDATL', 'RTO']
Unmapped areas: ['MIDATL', 'RTO']
Unmapped areas: ['RTO', 'MIDATL']
Unmapped areas: ['RTO', 'MIDATL']
Unmapped areas: ['RTO', 'MIDATL']
Unmapped areas: ['RTO', 'MIDATL']
Unmapped areas: ['MIDATL', 'RTO']
Unmapped areas: ['MIDATL', 'RTO']
Unmapped areas: ['MIDATL', 'RTO']


In [179]:
pjm_forecast = (
    pd.concat(forecast_profile_list, ignore_index=True)
    .sort_values('timestamp')
)
pjm_forecast.to_csv(f"../data/iso_load_profiles/pjm_forecast.csv", index=False)